# Q8-Q9: Gradient Descent Analysis

## Objectives
1. Compute condition number of Q = A^T A + λL^T L
2. Determine valid step size range for gradient descent
3. Estimate iterations needed for convergence
4. Compare gradient descent vs conjugate gradient

## Theory

### Quadratic Problem
The Tikhonov problem:
$$\min_x \frac{1}{2}\|Ax - y\|^2 + \frac{\lambda}{2}\|Lx\|^2$$

is equivalent to the quadratic:
$$\min_x \frac{1}{2}x^T Q x + b^T x$$

where:
- $Q = A^T A + \lambda L^T L$
- $b = -A^T y$

### Gradient Descent
Update: $x^{(k+1)} = x^{(k)} - \alpha \nabla f(x^{(k)}) = x^{(k)} - \alpha(Qx^{(k)} + b)$

### Step Size Constraint
For convergence: $0 < \alpha < \frac{2}{\lambda_{max}(Q)}$

### Convergence Rate
For optimal step size $\alpha^* = \frac{2}{\lambda_{min} + \lambda_{max}}$:
$$\frac{\|e^{(k+1)}\|}{\|e^{(k)}\|} \approx \frac{\kappa - 1}{\kappa + 1}$$

where $\kappa = \lambda_{max}/\lambda_{min}$ is the condition number.

Number of iterations for 10× reduction:
$$k \approx \frac{\log(0.1)}{2\log\left(\frac{\kappa-1}{\kappa+1}\right)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.data_utils import load_toy_measurements, measurements_to_vector
from src.matrix_construction import (
    build_toy_ray_path_matrix,
    build_combined_derivative_matrix
)
from src.solvers import gradient_descent, cgls

%matplotlib inline

## Setup: Build Q and b from Toy Problem

In [ ]:
# Load toy problem
Y = load_toy_measurements('../data/Y.mat')
y = measurements_to_vector(Y)

# Build matrices
M, N = 5, 5
A = build_toy_ray_path_matrix(M, N)
L = build_combined_derivative_matrix(M, N, dimensions=2)

# Regularization parameter
lam = 1e-5

print(f"A shape: {A.shape}")
print(f"L shape: {L.shape}")
print(f"λ = {lam}")

In [ ]:
# Build Q = A^T A + λ L^T L
# WARNING: Only do this for small problems! For large problems, never form Q explicitly.

from scipy import sparse

# Convert to sparse if needed
A_sparse = sparse.csr_matrix(A)

Q = A_sparse.T @ A_sparse + lam * (L.T @ L)
Q = Q.toarray()  # Convert to dense for eigenvalue computation

# Build b = -A^T y
b = -A_sparse.T @ y

print(f"Q shape: {Q.shape}")
print(f"Q is symmetric: {np.allclose(Q, Q.T)}")
print(f"b shape: {b.shape}")

## Q8: Compute Condition Number

In [ ]:
# Compute eigenvalues
eigenvalues = np.linalg.eigvalsh(Q)  # Use eigvalsh for symmetric matrices

lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]

print("Eigenvalue Analysis:")
print(f"  λ_min = {lambda_min:.6e}")
print(f"  λ_max = {lambda_max:.6e}")
print(f"  All eigenvalues > 0: {np.all(eigenvalues > 0)}")

# Condition number
kappa = lambda_max / lambda_min
print(f"\nCondition number κ(Q) = {kappa:.6e}")

In [ ]:
# Plot eigenvalue spectrum
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(eigenvalues, 'o-', markersize=4)
plt.xlabel('Index')
plt.ylabel('Eigenvalue')
plt.title('Eigenvalue Spectrum')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogy(eigenvalues, 'o-', markersize=4)
plt.xlabel('Index')
plt.ylabel('Eigenvalue (log scale)')
plt.title('Eigenvalue Spectrum (log scale)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/eigenvalue_spectrum.png', dpi=150)
plt.show()

## Step Size Analysis

In [ ]:
# Valid step size range
alpha_max = 2.0 / lambda_max
alpha_optimal = 2.0 / (lambda_min + lambda_max)

print("Step Size Analysis:")
print(f"  Valid range: 0 < α < {alpha_max:.6e}")
print(f"  Optimal α* = {alpha_optimal:.6e}")
print(f"  Safe choice: α = {0.9 * alpha_max:.6e} (90% of max)")

## Q9: Convergence Rate Estimate

In [ ]:
# Convergence rate
rho = (kappa - 1) / (kappa + 1)
print(f"Convergence rate ρ = {rho:.6f}")
print(f"This means error is multiplied by {rho:.6f} each iteration")

# Iterations for 10× reduction
import math
k_estimated = math.log(0.1) / (2 * math.log(rho))
print(f"\nEstimated iterations for 10× error reduction: {k_estimated:.1f}")

# For 100× reduction
k_100 = math.log(0.01) / (2 * math.log(rho))
print(f"Estimated iterations for 100× error reduction: {k_100:.1f}")

## Test Gradient Descent

In [ ]:
# Test with safe step size
step_size = 1.5 / lambda_max  # Safe choice

print(f"Running gradient descent with α = {step_size:.6e}...")
result_gd = gradient_descent(
    Q, b,
    step_size=step_size,
    max_iter=2000,
    tol=1e-8
)

print(f"\nGradient Descent Results:")
print(f"  Converged: {result_gd.converged}")
print(f"  Iterations: {result_gd.iterations}")
print(f"  Final objective: {result_gd.objective_values[-1]:.6e}")

## Compare with Conjugate Gradient (CGLS)

In [ ]:
# Solve same problem with CGLS
print("Running CGLS...")
result_cg = cgls(A, y, L, lam=lam, tol=1e-8, max_iter=2000)

print(f"\nCGLS Results:")
print(f"  Converged: {result_cg.converged}")
print(f"  Iterations: {result_cg.iterations}")
print(f"  Final objective: {result_cg.objective_values[-1]:.6e}")

## Convergence Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot residuals
ax1.semilogy(result_gd.residuals, 'b-', linewidth=2, label='Gradient Descent', alpha=0.7)
ax1.semilogy(result_cg.residuals, 'r--', linewidth=2, label='Conjugate Gradient', alpha=0.7)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Residual Norm')
ax1.set_title('Convergence: Residual Norm')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot objectives
ax2.plot(result_gd.objective_values, 'b-', linewidth=2, label='Gradient Descent', alpha=0.7)
ax2.plot(result_cg.objective_values, 'r--', linewidth=2, label='Conjugate Gradient', alpha=0.7)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Objective Value')
ax2.set_title('Convergence: Objective Function')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/gd_vs_cg_convergence.png', dpi=150)
plt.show()

print(f"\nSpeed-up factor: {result_gd.iterations / result_cg.iterations:.1f}×")

## Verify Theoretical Prediction

In [ ]:
# Find actual iteration where error reduced by 10×
residuals = np.array(result_gd.residuals)
initial_residual = residuals[0]
target_residual = initial_residual * 0.1

# Find first iteration below target
k_actual = np.where(residuals < target_residual)[0]
if len(k_actual) > 0:
    k_actual = k_actual[0]
else:
    k_actual = len(residuals)

print("Verification of Theoretical Estimate:")
print(f"  Estimated iterations for 10× reduction: {k_estimated:.1f}")
print(f"  Actual iterations: {k_actual}")
print(f"  Relative error: {abs(k_actual - k_estimated) / k_estimated * 100:.1f}%")

## Analysis

**Your answers here:**

### Q8: Condition Number
- κ(Q) = 
- This is [good/moderate/poor] conditioning because:
- Valid step size range:

### Q9: Convergence
- Estimated iterations for 10× reduction:
- Actual iterations:
- The estimate was [accurate/underestimate/overestimate]

### Why is Conjugate Gradient better?
- CG converges in at most n iterations (for n-dimensional problem)
- CG uses information from all previous iterations (Krylov subspace)
- GD only uses current gradient (steepest descent)
- For this problem, CG was __× faster


## Optional: Effect of λ on Condition Number

In [ ]:
# Test different λ values
lambda_values = [1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
condition_numbers = []

for lam in lambda_values:
    Q_temp = A_sparse.T @ A_sparse + lam * (L.T @ L)
    Q_temp = Q_temp.toarray()
    eigs = np.linalg.eigvalsh(Q_temp)
    kappa_temp = eigs[-1] / eigs[0]
    condition_numbers.append(kappa_temp)
    print(f"λ = {lam:.0e}: κ = {kappa_temp:.2e}")

# Plot
plt.figure(figsize=(8, 5))
plt.loglog(lambda_values, condition_numbers, 'o-', linewidth=2, markersize=8)
plt.xlabel('Regularization Parameter λ')
plt.ylabel('Condition Number κ(Q)')
plt.title('Effect of Regularization on Conditioning')
plt.grid(True, alpha=0.3)
plt.savefig('../results/figures/lambda_vs_condition_number.png', dpi=150)
plt.show()

print("\nObservation: Larger λ improves conditioning but may over-regularize!")